# EcoSignal Features 17–19 — Field Missions, Repeat-Scan Monitoring, and Guided Capture

This Colab notebook adds and tests three judge-visible systems without inventing scientific observations:

1. **Community Field Missions** convert the existing real-data Active Sampling Planner output into safe, explainable missions, estimate each mission's *heuristic* information gain, avoid redundant sampling locations, and maintain a hash-chained mission audit trail.
2. **Repeat-Scan Monitoring** links confirmed observations from the same tree/site, aligns real photographs with ORB + RANSAC, scores comparability, calculates uncertainty-aware coverage changes, and refuses to claim a trend when evidence is insufficient.
3. **Guided Capture** evaluates real phone/webcam photographs for blur, exposure, contrast, resolution, color clipping, and—when the final coverage model is connected—lichen framing, then returns specific retake instructions and an annotated overlay.

The notebook is compatible with the existing `EcoSignal_Complete_Features_01_16_Colab.ipynb` database and the real-map planner exports. It writes to a **separate prototype database** by default, so it cannot damage the production Features 1–16 database.

> Scientific boundary: expected confidence gain is a versioned prioritization heuristic, not a calibrated probability. Repeat-scan change intervals are screening estimates based on the held-out coverage MAE. Chlorosis is not used in these features.

In [ ]:
%pip install -q "numpy>=1.26,<3" "pandas>=2.2,<3" "Pillow>=10.4,<12" \
  "opencv-python-headless>=4.10,<5" "scikit-image>=0.24,<1" \
  "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" "folium>=0.17,<1" \
  "fastapi>=0.115,<1" "pydantic>=2.8,<3" "httpx>=0.27,<1" \
  "python-multipart>=0.0.9,<1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class EcoSignalPaths:
    root: Path = Path('/content/drive/MyDrive/EcoSignal')
    production_db: Path = Path('/content/drive/MyDrive/EcoSignal/data/ecosignal.db')
    output_dir: Path = Path('/content/drive/MyDrive/EcoSignal/features_17_19_outputs')
    coverage_model: Path = Path('/content/drive/MyDrive/EcoSignal/models/lichen_coverage_model.keras')
    repeat_manifest: Path = Path('/content/drive/MyDrive/EcoSignal/repeat_scan_manifest.csv')

PATHS = EcoSignalPaths()
PATHS.output_dir.mkdir(parents=True, exist_ok=True)

# Safety defaults. Integration with the production DB remains read-only in this notebook.
PROTOTYPE_DB = PATHS.output_dir / 'ecosignal_features_17_19_prototype.db'
AUTO_CONFIRM_GEOGRAPHIC_SITE_MATCHES = False
COVERAGE_THRESHOLD = 0.70
COVERAGE_HELDOUT_MAE_PP = 3.98

print('EcoSignal root:', PATHS.root)
print('Prototype outputs:', PATHS.output_dir)
print('Production DB exists:', PATHS.production_db.exists())

In [ ]:
import base64
import hashlib
import io
import json
import math
import re
import sqlite3
import tempfile
from datetime import datetime, timezone
from typing import Any, Iterable, Optional

import cv2
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Javascript, display
from PIL import Image, ImageDraw, ImageFont
from pydantic import BaseModel, Field
from sklearn.linear_model import LogisticRegression, TheilSenRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from skimage.metrics import structural_similarity

FEATURE_VERSIONS = {
    'field_missions': 'field_missions_v1.0.0',
    'mission_gain': 'mission_gain_heuristic_v1.0.0',
    'repeat_scan': 'repeat_scan_v1.0.0',
    'capture_guidance': 'capture_guidance_v1.0.0',
}

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def canonical_json(value: Any) -> str:
    return json.dumps(value, sort_keys=True, separators=(',', ':'), default=str)

def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def discover_file(root: Path, names: Iterable[str]) -> Optional[Path]:
    # Return the newest real file matching one of the supplied names.
    if not root.exists():
        return None
    matches = []
    for name in names:
        matches.extend(root.rglob(name))
    matches = [p for p in matches if p.is_file()]
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def safe_float(value: Any, default: float = np.nan) -> float:
    try:
        result = float(value)
        return result if np.isfinite(result) else default
    except (TypeError, ValueError):
        return default

def minmax(series: pd.Series, neutral: float = 0.5) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    lo, hi = values.min(), values.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return pd.Series(neutral, index=series.index, dtype=float)
    return ((values - lo) / (hi - lo)).clip(0, 1)

def haversine_matrix(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    lat_r = np.radians(np.asarray(lat, dtype=float))
    lon_r = np.radians(np.asarray(lon, dtype=float))
    dlat = lat_r[:, None] - lat_r[None, :]
    dlon = lon_r[:, None] - lon_r[None, :]
    a = np.sin(dlat / 2) ** 2 + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin(dlon / 2) ** 2
    return 6_371_000.0 * 2 * np.arctan2(np.sqrt(a), np.sqrt(np.maximum(1 - a, 0)))

## Real-input discovery

The notebook never creates fake map observations. It searches Drive for the planner's real `active_sampling_recommendations.csv` or GeoJSON. Repeat-scan and capture tests run only when real records/images are supplied; otherwise the final scorecard reports `NEEDS_REAL_INPUT` rather than a false pass.

In [ ]:
PLANNER_CSV = discover_file(PATHS.root.parent, ['active_sampling_recommendations.csv'])
PLANNER_GEOJSON = discover_file(PATHS.root.parent, ['active_sampling_recommendations.geojson'])

def load_planner_recommendations() -> tuple[pd.DataFrame, Optional[Path]]:
    if PLANNER_CSV:
        frame = pd.read_csv(PLANNER_CSV)
        source = PLANNER_CSV
    elif PLANNER_GEOJSON:
        payload = json.loads(PLANNER_GEOJSON.read_text())
        rows = []
        for feature in payload.get('features', []):
            props = dict(feature.get('properties') or {})
            coords = (feature.get('geometry') or {}).get('coordinates', [None, None])
            props.update({'longitude': coords[0], 'latitude': coords[1]})
            rows.append(props)
        frame, source = pd.DataFrame(rows), PLANNER_GEOJSON
    else:
        return pd.DataFrame(), None

    required = {'candidate_id', 'latitude', 'longitude', 'priority_score',
                'risk_score', 'uncertainty_score', 'access_status', 'objective'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Planner export is missing required columns: {sorted(missing)}')
    if not frame.latitude.between(-90, 90).all() or not frame.longitude.between(-180, 180).all():
        raise ValueError('Planner export contains invalid coordinates.')
    if frame.candidate_id.duplicated().any():
        raise ValueError('Planner candidate IDs must be unique.')
    return frame, source

planner_recommendations, planner_source = load_planner_recommendations()
if planner_source:
    print(f'Loaded {len(planner_recommendations)} real planner recommendations from {planner_source}')
else:
    print('NEEDS_REAL_INPUT: Run EcoSignal_Planner_and_Confidence_Layer_v2 first, then rerun this cell.')

# Feature 17 — Community Field Missions

The existing planner already chooses scientifically useful locations. This feature makes those locations actionable:

- estimates the marginal reduction in the planner's *heuristic* uncertainty surface using a spatial influence kernel;
- selects a diverse mission portfolio instead of repeatedly sampling neighboring points;
- respects `permission_required` and `field_verification_required` rather than claiming public access;
- creates field instructions, mission state transitions, and an append-only SHA-256 audit chain;
- exports frontend-ready JSON and GeoJSON.

In [ ]:
@dataclass(frozen=True)
class MissionConfig:
    expected_observation_quality: float = 0.82
    local_gain_cap: float = 0.55
    spatial_influence_m: float = 450.0
    redundancy_penalty: float = 18.0
    portfolio_size: int = 8
    minimum_spacing_m: float = 175.0
    permission_penalty: float = 0.35
    field_verification_penalty: float = 0.12

MISSION_CONFIG = MissionConfig()

ACCESS_FACTOR = {
    'public_access_indicated': 1.00,
    'public': 1.00,
    'field_verification_required': 0.72,
    'permission_required': 0.45,
}

def create_mission_candidates(frame: pd.DataFrame, cfg: MissionConfig = MISSION_CONFIG) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    out = frame.copy().reset_index(drop=True)
    blocked = out.access_status.astype(str).str.lower().isin({'restricted', 'excluded', 'unsafe'})
    out = out.loc[~blocked].reset_index(drop=True)
    for column in ['priority_score', 'risk_score', 'uncertainty_score', 'recent_change_score']:
        if column not in out:
            out[column] = 0.0
        out[column] = pd.to_numeric(out[column], errors='coerce').fillna(0.0)
    if out.priority_score.max() > 1.5:
        out['priority_norm'] = (out.priority_score / 100).clip(0, 1)
    else:
        out['priority_norm'] = out.priority_score.clip(0, 1)
    out['access_factor'] = out.access_status.map(ACCESS_FACTOR).fillna(0.55)
    out['scientific_value'] = (
        0.32 * out.uncertainty_score.clip(0, 1)
        + 0.24 * out.risk_score.clip(0, 1)
        + 0.18 * out.recent_change_score.clip(0, 1)
        + 0.16 * out.priority_norm
        + 0.10 * out.access_factor
    )

    distances = haversine_matrix(out.latitude.to_numpy(), out.longitude.to_numpy())
    kernel = np.exp(-0.5 * (distances / cfg.spatial_influence_m) ** 2)
    uncertainty = out.uncertainty_score.clip(0, 1).to_numpy()
    reductions = cfg.expected_observation_quality * kernel * uncertainty[None, :] * cfg.local_gain_cap
    out['estimated_local_confidence_gain_pp'] = 100 * cfg.expected_observation_quality * uncertainty * cfg.local_gain_cap
    out['estimated_portfolio_uncertainty_reduction_pp'] = 100 * reductions.mean(axis=1)
    gain_norm = minmax(out.estimated_portfolio_uncertainty_reduction_pp)
    out['mission_score'] = 100 * (0.68 * out.scientific_value + 0.22 * gain_norm + 0.10 * out.access_factor)
    out['mission_score'] = out.mission_score.clip(0, 100)
    out['mission_id'] = out.candidate_id.map(lambda x: 'MISSION-' + sha256_text(str(x))[:10].upper())
    out['gain_method'] = FEATURE_VERSIONS['mission_gain']
    return out

def select_diverse_missions(candidates: pd.DataFrame, cfg: MissionConfig = MISSION_CONFIG) -> pd.DataFrame:
    if candidates.empty:
        return candidates.copy()
    distances = haversine_matrix(candidates.latitude.to_numpy(), candidates.longitude.to_numpy())
    selected: list[int] = []
    remaining = set(range(len(candidates)))
    while remaining and len(selected) < min(cfg.portfolio_size, len(candidates)):
        best_idx, best_utility = None, -np.inf
        for idx in remaining:
            if selected:
                min_distance = float(distances[idx, selected].min())
                if min_distance < cfg.minimum_spacing_m:
                    continue
                redundancy = float(np.exp(-0.5 * (min_distance / cfg.spatial_influence_m) ** 2))
            else:
                redundancy = 0.0
            utility = float(candidates.iloc[idx].mission_score - cfg.redundancy_penalty * redundancy)
            if utility > best_utility:
                best_idx, best_utility = idx, utility
        if best_idx is None:
            break
        selected.append(best_idx)
        remaining.remove(best_idx)
    result = candidates.iloc[selected].copy().reset_index(drop=True)
    result.insert(0, 'mission_rank', np.arange(1, len(result) + 1))
    return result

def mission_instructions(row: pd.Series) -> dict[str, Any]:
    access = str(row.access_status)
    safety = {
        'permission_required': 'Do not enter until the property owner or site administrator approves access.',
        'field_verification_required': 'Confirm the location is publicly accessible and safe before approaching.',
    }.get(access, 'Remain on a safe public path and follow posted rules.')
    objective_text = {
        'control': 'Collect a stable control observation for comparison.',
        'hotspot_validation': 'Test whether the mapped environmental-risk hotspot is supported by a field observation.',
        'uncertainty_reduction': 'Fill a high-uncertainty gap in the community map.',
        'change_validation': 'Check an area where recent environmental change is elevated.',
    }.get(str(row.objective), 'Collect a standardized lichen observation.')
    return {
        'objective': objective_text,
        'safety_and_access': safety,
        'protocol': [
            'Confirm access and traffic conditions before entering the site.',
            'Choose a living tree with visible lichen at approximately chest height.',
            'Record the tree/site identifier so future scans can return to the same location.',
            'Use Guided Capture until the image passes or records a justified warning.',
            'Submit the original image, GPS accuracy, capture time, substrate, and scan result.',
        ],
        'scientific_contribution': (
            f"Estimated local heuristic-confidence gain: {row.estimated_local_confidence_gain_pp:.1f} percentage points. "
            'This is a prioritization estimate, not a guaranteed calibrated improvement.'
        ),
    }

def build_mission_packets(missions: pd.DataFrame) -> list[dict[str, Any]]:
    packets = []
    for _, row in missions.iterrows():
        packets.append({
            'mission_id': row.mission_id,
            'candidate_id': row.candidate_id,
            'rank': int(row.mission_rank),
            'status': 'available',
            'location': {'latitude': float(row.latitude), 'longitude': float(row.longitude)},
            'objective_type': str(row.objective),
            'site_type': str(row.get('site_type', 'unknown')),
            'access_status': str(row.access_status),
            'mission_score': round(float(row.mission_score), 2),
            'estimated_local_confidence_gain_pp': round(float(row.estimated_local_confidence_gain_pp), 2),
            'gain_method': str(row.gain_method),
            'instructions': mission_instructions(row),
            'planner_reason': str(row.get('reason', 'Generated from the EcoSignal real-data sampling planner.')),
            'created_at': utc_now_iso(),
        })
    return packets

In [ ]:
MISSION_TRANSITIONS = {
    'available': {'claimed'},
    'claimed': {'in_progress', 'available'},
    'in_progress': {'submitted', 'available'},
    'submitted': {'verified', 'rejected'},
    'rejected': {'in_progress'},
    'verified': set(),
}

def connect_mission_store(path: Path | str = PROTOTYPE_DB) -> sqlite3.Connection:
    connection = sqlite3.connect(str(path))
    connection.row_factory = sqlite3.Row
    connection.executescript('''
    CREATE TABLE IF NOT EXISTS field_missions (
      mission_id TEXT PRIMARY KEY,
      candidate_id TEXT NOT NULL,
      status TEXT NOT NULL,
      assigned_user_id TEXT,
      mission_json TEXT NOT NULL,
      created_at TEXT NOT NULL,
      updated_at TEXT NOT NULL
    );
    CREATE TABLE IF NOT EXISTS field_mission_events (
      event_id INTEGER PRIMARY KEY AUTOINCREMENT,
      mission_id TEXT NOT NULL,
      event_type TEXT NOT NULL,
      actor_id TEXT,
      payload_json TEXT NOT NULL,
      previous_hash TEXT,
      event_hash TEXT NOT NULL UNIQUE,
      created_at TEXT NOT NULL,
      FOREIGN KEY(mission_id) REFERENCES field_missions(mission_id)
    );
    ''')
    return connection

def append_mission_event(conn: sqlite3.Connection, mission_id: str, event_type: str,
                         actor_id: Optional[str], payload: dict[str, Any]) -> str:
    previous = conn.execute(
        'SELECT event_hash FROM field_mission_events WHERE mission_id=? ORDER BY event_id DESC LIMIT 1',
        (mission_id,),
    ).fetchone()
    previous_hash = previous['event_hash'] if previous else None
    created_at = utc_now_iso()
    body = {'mission_id': mission_id, 'event_type': event_type, 'actor_id': actor_id,
            'payload': payload, 'created_at': created_at, 'previous_hash': previous_hash}
    event_hash = sha256_text(canonical_json(body))
    conn.execute('''INSERT INTO field_mission_events
        (mission_id,event_type,actor_id,payload_json,previous_hash,event_hash,created_at)
        VALUES (?,?,?,?,?,?,?)''',
        (mission_id, event_type, actor_id, canonical_json(payload), previous_hash, event_hash, created_at))
    return event_hash

def sync_missions(conn: sqlite3.Connection, packets: list[dict[str, Any]]) -> None:
    now = utc_now_iso()
    for packet in packets:
        existing = conn.execute('SELECT mission_id FROM field_missions WHERE mission_id=?',
                                (packet['mission_id'],)).fetchone()
        if existing:
            conn.execute('UPDATE field_missions SET mission_json=?, updated_at=? WHERE mission_id=?',
                         (canonical_json(packet), now, packet['mission_id']))
        else:
            conn.execute('''INSERT INTO field_missions
                (mission_id,candidate_id,status,assigned_user_id,mission_json,created_at,updated_at)
                VALUES (?,?,?,?,?,?,?)''',
                (packet['mission_id'], packet['candidate_id'], 'available', None,
                 canonical_json(packet), now, now))
            append_mission_event(conn, packet['mission_id'], 'MISSION_CREATED', 'system',
                                 {'candidate_id': packet['candidate_id'], 'version': FEATURE_VERSIONS['field_missions']})
    conn.commit()

def transition_mission(conn: sqlite3.Connection, mission_id: str, new_status: str,
                       actor_id: str, payload: Optional[dict[str, Any]] = None) -> dict[str, Any]:
    row = conn.execute('SELECT * FROM field_missions WHERE mission_id=?', (mission_id,)).fetchone()
    if not row:
        raise KeyError(f'Unknown mission: {mission_id}')
    old_status = row['status']
    if new_status not in MISSION_TRANSITIONS.get(old_status, set()):
        raise ValueError(f'Invalid mission transition: {old_status} -> {new_status}')
    assigned = actor_id if new_status in {'claimed', 'in_progress', 'submitted'} else row['assigned_user_id']
    if new_status == 'available':
        assigned = None
    event_payload = {'from': old_status, 'to': new_status, **(payload or {})}
    conn.execute('UPDATE field_missions SET status=?, assigned_user_id=?, updated_at=? WHERE mission_id=?',
                 (new_status, assigned, utc_now_iso(), mission_id))
    append_mission_event(conn, mission_id, 'MISSION_STATUS_CHANGED', actor_id, event_payload)
    conn.commit()
    return {'mission_id': mission_id, 'previous_status': old_status,
            'status': new_status, 'assigned_user_id': assigned}

def verify_mission_chain(conn: sqlite3.Connection, mission_id: str) -> dict[str, Any]:
    rows = conn.execute('SELECT * FROM field_mission_events WHERE mission_id=? ORDER BY event_id',
                        (mission_id,)).fetchall()
    previous_hash = None
    for row in rows:
        payload = json.loads(row['payload_json'])
        body = {'mission_id': row['mission_id'], 'event_type': row['event_type'],
                'actor_id': row['actor_id'], 'payload': payload, 'created_at': row['created_at'],
                'previous_hash': row['previous_hash']}
        expected = sha256_text(canonical_json(body))
        if row['previous_hash'] != previous_hash or row['event_hash'] != expected:
            return {'valid': False, 'event_count': len(rows), 'failed_event_id': row['event_id']}
        previous_hash = row['event_hash']
    return {'valid': True, 'event_count': len(rows), 'latest_hash': previous_hash}

In [ ]:
mission_candidates = create_mission_candidates(planner_recommendations)
mission_portfolio = select_diverse_missions(mission_candidates)
mission_packets = build_mission_packets(mission_portfolio)

MISSION_JSON = PATHS.output_dir / 'field_missions.json'
MISSION_GEOJSON = PATHS.output_dir / 'field_missions.geojson'

if mission_packets:
    with connect_mission_store() as mission_conn:
        sync_missions(mission_conn, mission_packets)
        chain_results = {p['mission_id']: verify_mission_chain(mission_conn, p['mission_id'])
                         for p in mission_packets}
    MISSION_JSON.write_text(json.dumps({'version': FEATURE_VERSIONS['field_missions'],
                                        'missions': mission_packets}, indent=2))
    mission_features = [{
        'type': 'Feature',
        'geometry': {'type': 'Point', 'coordinates': [p['location']['longitude'], p['location']['latitude']]},
        'properties': {k: v for k, v in p.items() if k not in {'location', 'instructions'}},
    } for p in mission_packets]
    MISSION_GEOJSON.write_text(json.dumps({'type': 'FeatureCollection',
                                            'name': 'EcoSignal Community Field Missions',
                                            'features': mission_features}, indent=2))
    display(mission_portfolio[['mission_rank','mission_id','objective','site_type','access_status',
                               'mission_score','estimated_local_confidence_gain_pp']].round(2))
    print('Mission audit chains valid:', all(item['valid'] for item in chain_results.values()))
    print('Saved:', MISSION_JSON, MISSION_GEOJSON)
else:
    chain_results = {}
    print('NEEDS_REAL_INPUT: No mission portfolio created because no valid planner export was found.')

In [ ]:
def mission_map(missions: pd.DataFrame) -> Optional[folium.Map]:
    if missions.empty:
        return None
    center = [float(missions.latitude.mean()), float(missions.longitude.mean())]
    fmap = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
    colors = {'control':'blue', 'hotspot_validation':'red',
              'uncertainty_reduction':'orange', 'change_validation':'purple'}
    for row in missions.itertuples():
        popup = folium.Popup(
            f"<b>{row.mission_id}</b><br>Objective: {row.objective}<br>"
            f"Mission score: {row.mission_score:.1f}<br>"
            f"Estimated local gain: {row.estimated_local_confidence_gain_pp:.1f} pp<br>"
            f"Access: {row.access_status}", max_width=360)
        folium.Marker([row.latitude, row.longitude], popup=popup,
                      tooltip=f'#{row.mission_rank} {row.objective}',
                      icon=folium.Icon(color=colors.get(row.objective, 'green'), icon='leaf')).add_to(fmap)
    return fmap

field_mission_map = mission_map(mission_portfolio)
if field_mission_map:
    display(field_mission_map)

# Feature 18 — Repeat-Scan Environmental Change Tracking

A repeat scan is only scientifically useful if it refers to the same site and the photographs are reasonably comparable. This implementation therefore separates **site identity** from **visual comparability**:

- explicit `monitoring_site_id` is required for production comparisons;
- coordinates can propose a match, but the match remains unconfirmed by default;
- ORB features and RANSAC estimate geometric alignment;
- SSIM, exposure difference, overlap, and sharpness consistency create a comparability score;
- coverage change is accompanied by a conservative interval derived from held-out MAE and inflated when comparability is weak;
- trends require at least three dates and are estimated with Theil–Sen robust regression.

In [ ]:
REPEAT_COLUMNS = [
    'sample_id','monitoring_site_id','site_id_confirmed','site_match_method','site_match_distance_m','captured_at','image_path',
    'latitude','longitude','substrate_type','coverage_percent','coverage_confidence',
    'growth_form','growth_form_confidence','aqi','notes'
]

def parse_json_cell(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return value
    try:
        parsed = json.loads(value) if value else {}
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}

def extract_site_id(notes: Any) -> Optional[str]:
    match = re.search(r'(?:monitoring_)?site_id\s*[:=]\s*([A-Za-z0-9_.-]+)', str(notes or ''), re.I)
    return match.group(1) if match else None

def load_backend_monitoring_records(db_path: Path = PATHS.production_db) -> pd.DataFrame:
    if not db_path.exists():
        return pd.DataFrame(columns=REPEAT_COLUMNS)
    with sqlite3.connect(str(db_path)) as conn:
        table_names = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if not {'samples','predictions','sample_assets'} <= table_names:
            raise ValueError('EcoSignal database does not contain the expected Features 1–16 tables.')
        samples = pd.read_sql_query('SELECT * FROM samples', conn)
        predictions = pd.read_sql_query('SELECT * FROM predictions ORDER BY created_at', conn)
        assets = pd.read_sql_query("SELECT * FROM sample_assets WHERE asset_type='original_image' ORDER BY created_at", conn)

    latest_assets = assets.drop_duplicates('sample_id', keep='last').set_index('sample_id') if len(assets) else pd.DataFrame()
    rows = []
    for sample in samples.itertuples():
        row = {column: None for column in REPEAT_COLUMNS}
        row.update({'sample_id': sample.id, 'captured_at': sample.captured_at,
                    'latitude': sample.latitude, 'longitude': sample.longitude,
                    'substrate_type': sample.substrate_type, 'notes': sample.notes})
        explicit_site = extract_site_id(sample.notes)
        row['monitoring_site_id'] = explicit_site
        row['site_id_confirmed'] = bool(explicit_site)
        row['site_match_method'] = 'explicit_metadata' if explicit_site else None
        if len(assets) and sample.id in latest_assets.index:
            row['image_path'] = latest_assets.loc[sample.id, 'storage_path']
        sample_preds = predictions[predictions.sample_id == sample.id]
        for model_type in ('coverage','growth_form'):
            subset = sample_preds[sample_preds.model_type == model_type]
            if subset.empty:
                continue
            pred = subset.iloc[-1]
            value = parse_json_cell(pred.predicted_value_json)
            if model_type == 'coverage':
                row['coverage_percent'] = value.get('coverage_percent', value.get('value'))
                row['coverage_confidence'] = pred.confidence
            else:
                row['growth_form'] = value.get('class', value.get('growth_form'))
                row['growth_form_confidence'] = pred.confidence
        rows.append(row)
    return pd.DataFrame(rows, columns=REPEAT_COLUMNS)

def propose_geographic_site_matches(records: pd.DataFrame, radius_m: float = 4.0) -> pd.DataFrame:
    # Propose—never silently confirm—same-site groups using proximity and substrate.
    if records.empty:
        return records.copy()
    out = records.copy()
    eligible = out[
        out.monitoring_site_id.isna()
        & pd.to_numeric(out.latitude, errors='coerce').notna()
        & pd.to_numeric(out.longitude, errors='coerce').notna()
    ]
    for substrate, group in eligible.groupby(eligible.substrate_type.fillna('unknown')):
        indices = list(group.index)
        if len(indices) < 2:
            continue
        distance = haversine_matrix(group.latitude.astype(float).to_numpy(),
                                    group.longitude.astype(float).to_numpy())
        parent = list(range(len(indices)))
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]; x = parent[x]
            return x
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb: parent[rb] = ra
        for i in range(len(indices)):
            for j in range(i+1, len(indices)):
                if distance[i,j] <= radius_m: union(i,j)
        groups = {}
        for i, original_index in enumerate(indices):
            groups.setdefault(find(i), []).append((i, original_index))
        for members in groups.values():
            if len(members) < 2:
                continue
            sample_ids = sorted(str(out.loc[idx, 'sample_id']) for _, idx in members)
            candidate_id = 'GEO-CANDIDATE-' + sha256_text('|'.join(sample_ids))[:10].upper()
            for local_i, original_index in members:
                nearest = min(distance[local_i, other_i] for other_i, _ in members if other_i != local_i)
                out.loc[original_index, 'monitoring_site_id'] = candidate_id
                out.loc[original_index, 'site_id_confirmed'] = AUTO_CONFIRM_GEOGRAPHIC_SITE_MATCHES
                out.loc[original_index, 'site_match_method'] = 'geo_candidate_requires_review'
                out.loc[original_index, 'site_match_distance_m'] = round(float(nearest), 2)
    return out

def load_repeat_manifest() -> pd.DataFrame:
    if PATHS.repeat_manifest.exists():
        manifest = pd.read_csv(PATHS.repeat_manifest)
        missing = set(REPEAT_COLUMNS) - set(manifest.columns)
        for column in missing:
            manifest[column] = None
        manifest['site_id_confirmed'] = manifest.site_id_confirmed.astype(str).str.lower().isin({'true','1','yes'})
        return manifest[REPEAT_COLUMNS]
    backend = propose_geographic_site_matches(load_backend_monitoring_records())
    if not backend.empty:
        template_path = PATHS.output_dir / 'repeat_scan_manifest_TO_REVIEW.csv'
        backend.to_csv(template_path, index=False)
        print('Created a review template. Assign the same monitoring_site_id only to confirmed same-tree/site scans:')
        print(template_path)
    return backend

repeat_records = load_repeat_manifest()
print('Repeat-scan records discovered:', len(repeat_records))
print('Confirmed site IDs:', int(repeat_records.site_id_confirmed.fillna(False).sum()) if len(repeat_records) else 0)

In [ ]:
def load_cv_image(path: str | Path, max_dimension: int = 1400) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'Could not read image: {path}')
    height, width = image.shape[:2]
    scale = min(1.0, max_dimension / max(height, width))
    if scale < 1:
        image = cv2.resize(image, (round(width * scale), round(height * scale)), interpolation=cv2.INTER_AREA)
    return image

def normalized_luminance(image_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

def image_sharpness(image_bgr: np.ndarray) -> float:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def align_repeat_images(baseline_path: str | Path, current_path: str | Path) -> dict[str, Any]:
    base = load_cv_image(baseline_path)
    current = load_cv_image(current_path)
    current = cv2.resize(current, (base.shape[1], base.shape[0]), interpolation=cv2.INTER_AREA)
    base_gray, current_gray = normalized_luminance(base), normalized_luminance(current)
    orb = cv2.ORB_create(nfeatures=3500, scaleFactor=1.2, nlevels=8)
    key1, desc1 = orb.detectAndCompute(base_gray, None)
    key2, desc2 = orb.detectAndCompute(current_gray, None)
    if desc1 is None or desc2 is None or len(key1) < 12 or len(key2) < 12:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'Insufficient stable visual features for same-site alignment.',
                'metrics':{'keypoints_baseline':len(key1),'keypoints_current':len(key2)}}
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    pairs = matcher.knnMatch(desc2, desc1, k=2)
    good = [m for m, n in pairs if m.distance < 0.72 * n.distance]
    if len(good) < 12:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'Too few consistent feature matches; reproduce the original framing.',
                'metrics':{'good_matches':len(good)}}
    src = np.float32([key2[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([key1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    homography, inliers = cv2.findHomography(src, dst, cv2.RANSAC, 4.0)
    if homography is None or inliers is None:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'RANSAC could not establish a stable geometric alignment.',
                'metrics':{'good_matches':len(good)}}
    aligned = cv2.warpPerspective(current, homography, (base.shape[1], base.shape[0]))
    valid = cv2.warpPerspective(np.full(current.shape[:2], 255, np.uint8), homography,
                                (base.shape[1], base.shape[0])) > 0
    overlap = float(valid.mean())
    inlier_ratio = float(inliers.ravel().mean())
    ys, xs = np.where(valid)
    if len(xs) < 100:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'Aligned image overlap is too small.', 'metrics':{'overlap_fraction':overlap}}
    x0, x1, y0, y1 = xs.min(), xs.max()+1, ys.min(), ys.max()+1
    a = normalized_luminance(base)[y0:y1, x0:x1]
    b = normalized_luminance(aligned)[y0:y1, x0:x1]
    ssim = float(structural_similarity(a, b, data_range=255))
    exposure_difference = abs(float(cv2.cvtColor(base, cv2.COLOR_BGR2GRAY).mean())
                              - float(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)[valid].mean())) / 255
    sharp_ratio = min(image_sharpness(base), image_sharpness(aligned)) / max(image_sharpness(base), image_sharpness(aligned), 1e-6)
    components = {
        'ransac_inlier_ratio': np.clip(inlier_ratio / 0.65, 0, 1),
        'structural_similarity': np.clip((ssim - 0.15) / 0.70, 0, 1),
        'exposure_consistency': np.clip(1 - exposure_difference / 0.35, 0, 1),
        'overlap_fraction': np.clip(overlap / 0.85, 0, 1),
        'sharpness_consistency': np.clip(sharp_ratio, 0, 1),
    }
    score = float(0.32*components['ransac_inlier_ratio'] + 0.30*components['structural_similarity']
                  + 0.16*components['exposure_consistency'] + 0.12*components['overlap_fraction']
                  + 0.10*components['sharpness_consistency'])
    decision = 'accepted' if score >= 0.72 else ('caution' if score >= 0.52 else 'retake_required')
    reason = {'accepted':'Images are sufficiently comparable for screening-level change analysis.',
              'caution':'Images can be compared only with increased uncertainty.',
              'retake_required':'Framing or visual conditions differ too much for a defensible comparison.'}[decision]
    return {'decision':decision, 'comparability_score':round(score, 4), 'reason':reason,
            'metrics':{**{k:round(float(v),4) for k,v in components.items()},
                       'raw_ssim':round(ssim,4),'raw_inlier_ratio':round(inlier_ratio,4),
                       'raw_overlap_fraction':round(overlap,4),
                       'exposure_difference':round(exposure_difference,4),
                       'good_matches':len(good)},
            '_aligned_image':aligned, '_baseline_image':base}

In [ ]:
def coverage_change_interval(previous: float, current: float, comparability_score: float,
                             mae_pp: float = COVERAGE_HELDOUT_MAE_PP) -> dict[str, Any]:
    delta = float(current - previous)
    # Normal-residual approximation: sigma ~= MAE * sqrt(pi/2). Difference combines two measurements.
    residual_sigma = mae_pp * math.sqrt(math.pi / 2)
    base_half_width = 1.96 * math.sqrt(2) * residual_sigma
    inflation = 1 / max(float(comparability_score), 0.45)
    half_width = base_half_width * inflation
    lower, upper = delta - half_width, delta + half_width
    if lower > 0:
        status = 'increase_supported'
    elif upper < 0:
        status = 'decrease_supported'
    else:
        status = 'no_reliable_change_detected'
    return {'coverage_delta_pp':round(delta,2), 'approx_95_interval_pp':[round(lower,2),round(upper,2)],
            'change_status':status, 'mae_basis_pp':mae_pp,
            'interval_method':'MAE-normal approximation, comparability-inflated; screening only'}

def repeat_scan_pairs(records: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    if records.empty:
        return pd.DataFrame(), []
    usable = records.copy()
    usable = usable[usable.site_id_confirmed.fillna(False).astype(bool)]
    usable = usable[usable.monitoring_site_id.notna() & usable.captured_at.notna() & usable.image_path.notna()]
    usable['captured_at'] = pd.to_datetime(usable.captured_at, errors='coerce', utc=True)
    usable = usable.dropna(subset=['captured_at'])
    results = []
    for site_id, group in usable.groupby('monitoring_site_id'):
        group = group.sort_values('captured_at')
        for previous_index, current_index in zip(group.index[:-1], group.index[1:]):
            previous, current = group.loc[previous_index], group.loc[current_index]
            if not Path(str(previous.image_path)).exists() or not Path(str(current.image_path)).exists():
                results.append({'monitoring_site_id':site_id, 'previous_sample_id':previous.sample_id,
                                'current_sample_id':current.sample_id, 'status':'missing_image_file'})
                continue
            alignment = align_repeat_images(previous.image_path, current.image_path)
            result = {'monitoring_site_id':site_id, 'previous_sample_id':previous.sample_id,
                      'current_sample_id':current.sample_id, 'previous_captured_at':previous.captured_at.isoformat(),
                      'current_captured_at':current.captured_at.isoformat(),
                      'comparability':{k:v for k,v in alignment.items() if not k.startswith('_')},
                      'version':FEATURE_VERSIONS['repeat_scan']}
            prev_cov, curr_cov = safe_float(previous.coverage_percent), safe_float(current.coverage_percent)
            if np.isfinite(prev_cov) and np.isfinite(curr_cov) and alignment['decision'] != 'retake_required':
                result['coverage_change'] = coverage_change_interval(prev_cov, curr_cov,
                                                                     alignment['comparability_score'])
            else:
                result['coverage_change'] = {'change_status':'unavailable',
                                             'reason':'Coverage values are missing or images require a retake.'}
            prev_growth, curr_growth = str(previous.growth_form or ''), str(current.growth_form or '')
            prev_gc, curr_gc = safe_float(previous.growth_form_confidence), safe_float(current.growth_form_confidence)
            result['growth_form_check'] = {
                'previous':prev_growth or None, 'current':curr_growth or None,
                'both_high_confidence':bool(prev_gc >= 0.80 and curr_gc >= 0.80),
                'class_changed':bool(prev_growth and curr_growth and prev_growth != curr_growth),
                'interpretation':'classifier_consistency_flag_not_ecological_trend',
            }
            results.append(result)
    return usable, results

def site_trends(records: pd.DataFrame) -> list[dict[str, Any]]:
    trends = []
    for site_id, group in records.groupby('monitoring_site_id') if not records.empty else []:
        group = group.copy().sort_values('captured_at')
        group['coverage_percent'] = pd.to_numeric(group.coverage_percent, errors='coerce')
        group = group.dropna(subset=['coverage_percent'])
        if len(group) < 3:
            trends.append({'monitoring_site_id':site_id, 'status':'insufficient_dates', 'n_dates':len(group)})
            continue
        days = (group.captured_at - group.captured_at.min()).dt.total_seconds().to_numpy() / 86400
        if days.max() < 30:
            trends.append({'monitoring_site_id':site_id, 'status':'insufficient_time_span', 'n_dates':len(group)})
            continue
        model = TheilSenRegressor(random_state=42).fit(days.reshape(-1,1), group.coverage_percent.to_numpy())
        trends.append({'monitoring_site_id':site_id, 'status':'screening_trend_available',
                       'n_dates':len(group), 'days_observed':round(float(days.max()),1),
                       'coverage_slope_pp_per_year':round(float(model.coef_[0]*365.25),2),
                       'interpretation':'robust descriptive trend; causal attribution is not supported'})
    return trends

In [ ]:
confirmed_repeat_records, repeat_results = repeat_scan_pairs(repeat_records)
repeat_trends = site_trends(confirmed_repeat_records)
REPEAT_RESULTS_PATH = PATHS.output_dir / 'repeat_scan_results.json'
REPEAT_RESULTS_PATH.write_text(json.dumps({'version':FEATURE_VERSIONS['repeat_scan'],
                                            'pair_results':repeat_results,
                                            'site_trends':repeat_trends}, indent=2))
if repeat_results:
    summary_rows = []
    for item in repeat_results:
        summary_rows.append({
            'site_id':item.get('monitoring_site_id'),
            'previous':item.get('previous_sample_id'),
            'current':item.get('current_sample_id'),
            'comparability':(item.get('comparability') or {}).get('decision'),
            'score':(item.get('comparability') or {}).get('comparability_score'),
            'coverage_change':(item.get('coverage_change') or {}).get('change_status'),
        })
    display(pd.DataFrame(summary_rows))
else:
    print('NEEDS_REAL_INPUT: Confirm at least two real scans of the same tree/site in repeat_scan_manifest.csv.')
print('Saved:', REPEAT_RESULTS_PATH)

# Feature 19 — Intelligent Guided Capture

Guided Capture works in two layers. The general image-quality layer is available immediately. The lichen-framing layer activates only when the finalized coverage model is found and successfully loaded. The default coverage threshold is the validated **0.70**.

The output is designed for the frontend: an overall decision, normalized component scores, raw metrics, ordered retake instructions, provenance/version fields, and an annotated overlay.

In [ ]:
@dataclass(frozen=True)
class CaptureThresholds:
    min_width: int = 640
    min_height: int = 640
    min_blur_variance: float = 85.0
    min_mean_luminance: float = 0.18
    max_mean_luminance: float = 0.86
    min_robust_contrast: float = 0.20
    max_underexposed_fraction: float = 0.22
    max_overexposed_fraction: float = 0.18
    min_lichen_fraction: float = 0.08
    max_lichen_fraction: float = 0.92
    max_centroid_offset: float = 0.34
    max_border_touch_sides: int = 3

CAPTURE_THRESHOLDS = CaptureThresholds()
_coverage_model = None

def locate_coverage_model() -> Optional[Path]:
    if PATHS.coverage_model.exists():
        return PATHS.coverage_model
    return discover_file(PATHS.root.parent, ['lichen_coverage_model.keras', 'lichen_coverage_model(1).keras'])

def load_coverage_model():
    global _coverage_model
    if _coverage_model is not None:
        return _coverage_model
    model_path = locate_coverage_model()
    if model_path is None:
        return None
    try:
        import tensorflow as tf
        _coverage_model = tf.keras.models.load_model(model_path, compile=False)
        print('Loaded coverage model:', model_path)
        return _coverage_model
    except Exception as exc:
        print('Coverage model not connected; general quality guidance remains active:', exc)
        return None

def predict_coverage_mask(image_rgb: np.ndarray, threshold: float = COVERAGE_THRESHOLD) -> Optional[np.ndarray]:
    model = load_coverage_model()
    if model is None:
        return None
    input_shape = model.input_shape
    height = int(input_shape[1] or 256)
    width = int(input_shape[2] or 256)
    resized = cv2.resize(image_rgb, (width, height), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    prediction = np.asarray(model.predict(resized[None, ...], verbose=0))
    prediction = np.squeeze(prediction)
    if prediction.ndim == 3:
        prediction = prediction[..., 0]
    probability = cv2.resize(prediction.astype(np.float32), (image_rgb.shape[1], image_rgb.shape[0]),
                             interpolation=cv2.INTER_LINEAR)
    mask = (probability >= threshold).astype(np.uint8)
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def stable_quality_image(image_rgb: np.ndarray, max_dimension: int = 900) -> np.ndarray:
    h, w = image_rgb.shape[:2]
    scale = min(1.0, max_dimension / max(h, w))
    return cv2.resize(image_rgb, (round(w*scale), round(h*scale)), interpolation=cv2.INTER_AREA) if scale < 1 else image_rgb

def base_capture_metrics(image_rgb: np.ndarray) -> dict[str, float]:
    stable = stable_quality_image(image_rgb)
    gray = cv2.cvtColor(stable, cv2.COLOR_RGB2GRAY)
    luma = gray.astype(np.float32) / 255
    p05, p95 = np.percentile(luma, [5, 95])
    hsv = cv2.cvtColor(stable, cv2.COLOR_RGB2HSV)
    return {
        'width':int(image_rgb.shape[1]), 'height':int(image_rgb.shape[0]),
        'blur_variance':float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        'mean_luminance':float(luma.mean()),
        'robust_contrast':float(p95-p05),
        'underexposed_fraction':float((luma < 0.05).mean()),
        'overexposed_fraction':float((luma > 0.98).mean()),
        'mean_saturation':float((hsv[...,1].astype(np.float32)/255).mean()),
    }

def mask_framing_metrics(mask: np.ndarray) -> dict[str, Any]:
    binary = (mask > 0).astype(np.uint8)
    fraction = float(binary.mean())
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, 8)
    if count <= 1 or binary.sum() == 0:
        return {'lichen_fraction':fraction, 'largest_component_fraction':0.0,
                'centroid_offset':1.0, 'border_touch_sides':0}
    component = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    largest = labels == component
    cx, cy = centroids[component]
    center_x, center_y = (mask.shape[1]-1)/2, (mask.shape[0]-1)/2
    diagonal_half = math.hypot(center_x, center_y)
    offset = math.hypot(cx-center_x, cy-center_y) / max(diagonal_half, 1)
    sides = int(largest[0,:].any()) + int(largest[-1,:].any()) + int(largest[:,0].any()) + int(largest[:,-1].any())
    return {'lichen_fraction':fraction,
            'largest_component_fraction':float(largest.sum()/max(binary.sum(),1)),
            'centroid_offset':float(offset), 'border_touch_sides':sides}

def assess_capture(image_path: str | Path, thresholds: CaptureThresholds = CAPTURE_THRESHOLDS,
                   use_coverage_model: bool = True) -> dict[str, Any]:
    image = np.asarray(Image.open(image_path).convert('RGB'))
    metrics = base_capture_metrics(image)
    component_scores = {
        'resolution':min(1.0, metrics['width']/thresholds.min_width, metrics['height']/thresholds.min_height),
        'sharpness':float(np.clip(metrics['blur_variance']/thresholds.min_blur_variance, 0, 1)),
        'brightness':float(np.clip(1-abs(metrics['mean_luminance']-0.52)/0.42, 0, 1)),
        'contrast':float(np.clip(metrics['robust_contrast']/thresholds.min_robust_contrast, 0, 1)),
        'exposure_clipping':float(np.clip(1-max(metrics['underexposed_fraction']/thresholds.max_underexposed_fraction,
                                                metrics['overexposed_fraction']/thresholds.max_overexposed_fraction),0,1)),
    }
    instructions = []
    critical = []
    if metrics['width'] < thresholds.min_width or metrics['height'] < thresholds.min_height:
        critical.append('IMAGE_TOO_SMALL'); instructions.append('Move closer or use the full-resolution camera mode.')
    if metrics['blur_variance'] < thresholds.min_blur_variance:
        critical.append('IMAGE_TOO_BLURRY'); instructions.append('Hold the phone steady, tap to focus, and retake.')
    if metrics['mean_luminance'] < thresholds.min_mean_luminance:
        critical.append('IMAGE_TOO_DARK'); instructions.append('Add indirect light or move out of deep shade.')
    elif metrics['mean_luminance'] > thresholds.max_mean_luminance:
        critical.append('IMAGE_TOO_BRIGHT'); instructions.append('Reduce glare and avoid direct flash or harsh sunlight.')
    if metrics['robust_contrast'] < thresholds.min_robust_contrast:
        critical.append('LOW_CONTRAST'); instructions.append('Improve focus and use even side lighting.')
    if metrics['overexposed_fraction'] > thresholds.max_overexposed_fraction:
        critical.append('EXCESSIVE_HIGHLIGHT_CLIPPING'); instructions.append('Tilt the camera to remove glare.')

    mask = predict_coverage_mask(image) if use_coverage_model else None
    if mask is not None:
        framing = mask_framing_metrics(mask)
        metrics.update(framing)
        fraction = framing['lichen_fraction']
        component_scores['lichen_presence'] = float(np.clip(fraction/thresholds.min_lichen_fraction, 0, 1))
        component_scores['framing'] = float(np.clip(1-framing['centroid_offset']/thresholds.max_centroid_offset,0,1))
        if fraction < thresholds.min_lichen_fraction:
            critical.append('TOO_LITTLE_LICHEN'); instructions.append('Move closer and center a larger lichen-covered area.')
        elif fraction > thresholds.max_lichen_fraction:
            instructions.append('Pull back slightly so the scan includes the lichen boundary and some context.')
        if framing['centroid_offset'] > thresholds.max_centroid_offset:
            instructions.append('Recenter the lichen inside the guide frame.')
        if framing['border_touch_sides'] > thresholds.max_border_touch_sides:
            instructions.append('Pull back slightly; the detected lichen is cut off on most sides.')
        model_status = 'coverage_model_connected'
    else:
        model_status = 'general_quality_only'

    weights = {'resolution':0.14,'sharpness':0.24,'brightness':0.18,'contrast':0.16,'exposure_clipping':0.13,
               'lichen_presence':0.08,'framing':0.07}
    active_weight = sum(weights[k] for k in component_scores)
    overall = sum(weights[k]*component_scores[k] for k in component_scores)/active_weight
    if critical:
        decision = 'retake_required'
    elif overall >= 0.78:
        decision = 'accepted'
    elif overall >= 0.62:
        decision = 'accepted_with_caution'
    else:
        decision = 'retake_required'
    if not instructions and decision == 'accepted':
        instructions = ['Capture quality passed. Keep this framing for future repeat scans.']
    return {'feature':'guided_capture', 'version':FEATURE_VERSIONS['capture_guidance'],
            'decision':decision, 'overall_score':round(float(overall),4),
            'model_status':model_status, 'coverage_threshold':COVERAGE_THRESHOLD if mask is not None else None,
            'metrics':{k:round(float(v),4) if isinstance(v,(float,np.floating)) else int(v) for k,v in metrics.items()},
            'component_scores':{k:round(float(v),4) for k,v in component_scores.items()},
            'warning_codes':critical, 'instructions':instructions,
            '_mask':mask, '_image':image}

def save_capture_assessment(image_path: str | Path, result: dict[str, Any]) -> tuple[Path, Path]:
    stem = Path(image_path).stem
    json_path = PATHS.output_dir / f'{stem}_capture_assessment.json'
    overlay_path = PATHS.output_dir / f'{stem}_capture_overlay.png'
    serializable = {k:v for k,v in result.items() if not k.startswith('_')}
    json_path.write_text(json.dumps(serializable, indent=2))
    image = Image.fromarray(result['_image']).convert('RGB')
    draw = ImageDraw.Draw(image, 'RGBA')
    w, h = image.size
    color = (32,190,110,255) if result['decision']=='accepted' else (239,78,78,255)
    margin_x, margin_y = int(w*0.12), int(h*0.12)
    draw.rounded_rectangle((margin_x,margin_y,w-margin_x,h-margin_y), radius=max(8,int(min(w,h)*0.02)),
                           outline=color, width=max(4,int(min(w,h)*0.008)))
    mask = result.get('_mask')
    if mask is not None:
        overlay = np.zeros((h,w,4), np.uint8); overlay[mask>0] = (30,190,110,75)
        image = Image.alpha_composite(image.convert('RGBA'), Image.fromarray(overlay, 'RGBA')).convert('RGB')
    image.save(overlay_path)
    return json_path, overlay_path

In [ ]:
# Optional Colab webcam capture with a composition guide. Run this cell, then call:
# captured_path = take_guided_photo('lichen_capture.jpg')
from google.colab.output import eval_js

def take_guided_photo(filename: str = 'lichen_capture.jpg', quality: float = 0.92) -> Path:
    script = r'''
    async function captureEcoSignal(quality) {
      const container = document.createElement('div');
      const video = document.createElement('video');
      const canvas = document.createElement('canvas');
      const button = document.createElement('button');
      const note = document.createElement('div');
      button.textContent = 'Capture lichen photo';
      note.textContent = 'Center the lichen inside the green frame; avoid glare and hold steady.';
      note.style = 'font: 14px sans-serif; margin:8px 0;';
      video.style = 'width:100%;max-width:720px;border:3px solid #2fbf71;border-radius:12px;';
      container.append(note, video, button); document.body.appendChild(container);
      const stream = await navigator.mediaDevices.getUserMedia({video:{facingMode:'environment'}});
      video.srcObject = stream; await video.play();
      await new Promise(resolve => button.onclick = resolve);
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      const ctx = canvas.getContext('2d'); ctx.drawImage(video,0,0);
      stream.getTracks().forEach(track => track.stop()); container.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }'''
    data = eval_js(script + f'captureEcoSignal({quality})')
    binary = base64.b64decode(data.split(',')[1])
    path = PATHS.output_dir / filename
    path.write_bytes(binary)
    return path

print('Webcam helper ready. It captures a real image; no example image is generated.')

In [ ]:
# Point this variable to a real phone photo, or assign the output of take_guided_photo().
REAL_CAPTURE_IMAGE_PATH = ''

capture_result = None
capture_json_path = None
capture_overlay_path = None
if REAL_CAPTURE_IMAGE_PATH and Path(REAL_CAPTURE_IMAGE_PATH).exists():
    capture_result = assess_capture(REAL_CAPTURE_IMAGE_PATH)
    capture_json_path, capture_overlay_path = save_capture_assessment(REAL_CAPTURE_IMAGE_PATH, capture_result)
    display(Image.open(capture_overlay_path))
    print(json.dumps({k:v for k,v in capture_result.items() if not k.startswith('_')}, indent=2))
    print('Saved:', capture_json_path, capture_overlay_path)
else:
    print("NEEDS_REAL_INPUT: Set REAL_CAPTURE_IMAGE_PATH or run captured_path = take_guided_photo(); then set REAL_CAPTURE_IMAGE_PATH = str(captured_path).")

## Optional real-photo calibration

Place photos you judged acceptable in `EcoSignal/capture_calibration/accepted/` and rejected photos in `.../rejected/`. With at least 12 real images per group, this cell estimates how well the quality metrics separate your actual phone/field conditions. It reports cross-validated ROC-AUC but does **not** silently replace the thresholds; review failure galleries before changing production settings.

In [ ]:
CALIBRATION_ROOT = PATHS.root / 'capture_calibration'

def calibration_features(path: Path) -> dict[str, Any]:
    image = np.asarray(Image.open(path).convert('RGB'))
    metrics = base_capture_metrics(image)
    return {'image_path':str(path), **metrics}

def run_capture_calibration(root: Path = CALIBRATION_ROOT) -> dict[str, Any]:
    accepted = [p for p in (root/'accepted').glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}] if (root/'accepted').exists() else []
    rejected = [p for p in (root/'rejected').glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}] if (root/'rejected').exists() else []
    if len(accepted) < 12 or len(rejected) < 12:
        return {'status':'NEEDS_REAL_INPUT','accepted_images':len(accepted),'rejected_images':len(rejected),
                'requirement':'At least 12 real images per group.'}
    rows = [{**calibration_features(p),'label':1} for p in accepted] + [{**calibration_features(p),'label':0} for p in rejected]
    frame = pd.DataFrame(rows)
    feature_names = ['blur_variance','mean_luminance','robust_contrast','underexposed_fraction',
                     'overexposed_fraction','mean_saturation']
    X = frame[feature_names].replace([np.inf,-np.inf],np.nan).fillna(0)
    y = frame.label
    folds = max(3, min(5, int(min(y.value_counts()))))
    cv = StratifiedKFold(folds, shuffle=True, random_state=42)
    model = LogisticRegression(max_iter=2000, class_weight='balanced')
    probability = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:,1]
    auc = roc_auc_score(y, probability)
    model.fit(X,y)
    frame['quality_probability_oof'] = probability
    output = PATHS.output_dir / 'capture_calibration_results.csv'; frame.to_csv(output,index=False)
    return {'status':'PASS' if auc >= 0.75 else 'REVIEW_NEEDED', 'cross_validated_auc':round(float(auc),3),
            'accepted_images':len(accepted),'rejected_images':len(rejected),
            'coefficients':dict(zip(feature_names, model.coef_[0].round(4))), 'output':str(output)}

capture_calibration = run_capture_calibration()
print(json.dumps(capture_calibration, indent=2))

# Integration-ready service contracts and local API tests

The notebook exposes the three features through a small FastAPI surface compatible with the existing backend style. The local tests use real planner records and real images only; unavailable inputs are marked as skipped.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

class MissionTransitionRequest(BaseModel):
    actor_id: str = Field(min_length=1)
    new_status: str
    sample_id: Optional[str] = None
    asset_checksum: Optional[str] = None

class RepeatCompareRequest(BaseModel):
    baseline_image_path: str
    current_image_path: str
    previous_coverage_percent: Optional[float] = None
    current_coverage_percent: Optional[float] = None

class CaptureAssessRequest(BaseModel):
    image_path: str
    use_coverage_model: bool = True

feature_app = FastAPI(title='EcoSignal Features 17–19 Prototype', version='1.0.0')

@feature_app.get('/feature-health')
def feature_health():
    return {'status':'ok','versions':FEATURE_VERSIONS,
            'real_missions_loaded':len(mission_packets),
            'repeat_pairs_evaluated':len(repeat_results),
            'coverage_model_found':locate_coverage_model() is not None}

@feature_app.get('/field-missions')
def list_field_missions(status: Optional[str] = None):
    with connect_mission_store() as conn:
        query, params = 'SELECT * FROM field_missions', []
        if status:
            query += ' WHERE status=?'; params.append(status)
        rows = conn.execute(query, params).fetchall()
    return [{**json.loads(row['mission_json']), 'status':row['status'],
             'assigned_user_id':row['assigned_user_id']} for row in rows]

@feature_app.post('/field-missions/{mission_id}/transition')
def mission_transition(mission_id: str, request: MissionTransitionRequest):
    payload = {k:v for k,v in {'sample_id':request.sample_id,
                                'asset_checksum':request.asset_checksum}.items() if v is not None}
    try:
        with connect_mission_store() as conn:
            return transition_mission(conn, mission_id, request.new_status, request.actor_id, payload)
    except KeyError as exc:
        raise HTTPException(404, str(exc))
    except ValueError as exc:
        raise HTTPException(409, str(exc))

@feature_app.post('/repeat-scan/compare')
def repeat_compare(request: RepeatCompareRequest):
    for path in (request.baseline_image_path, request.current_image_path):
        if not Path(path).exists():
            raise HTTPException(404, f'Image not found: {path}')
    result = align_repeat_images(request.baseline_image_path, request.current_image_path)
    public = {k:v for k,v in result.items() if not k.startswith('_')}
    if request.previous_coverage_percent is not None and request.current_coverage_percent is not None and result['decision'] != 'retake_required':
        public['coverage_change'] = coverage_change_interval(request.previous_coverage_percent,
                                                             request.current_coverage_percent,
                                                             result['comparability_score'])
    return public

@feature_app.post('/guided-capture/assess')
def guided_capture_assess(request: CaptureAssessRequest):
    if not Path(request.image_path).exists():
        raise HTTPException(404, f'Image not found: {request.image_path}')
    result = assess_capture(request.image_path, use_coverage_model=request.use_coverage_model)
    return {k:v for k,v in result.items() if not k.startswith('_')}

with TestClient(feature_app) as api_client:
    health_response = api_client.get('/feature-health')
    assert health_response.status_code == 200
    assert health_response.json()['status'] == 'ok'
    if mission_packets:
        missions_response = api_client.get('/field-missions')
        assert missions_response.status_code == 200 and len(missions_response.json()) >= 1
print('PASS: Feature service contracts and non-mutating API smoke tests succeeded.')

In [ ]:
def dry_run_mission_workflow(packets: list[dict[str, Any]]) -> dict[str, Any]:
    if not packets:
        return {'status':'NEEDS_REAL_INPUT','reason':'No real planner mission available.'}
    with tempfile.TemporaryDirectory() as temp_dir:
        with connect_mission_store(Path(temp_dir)/'mission_test.db') as conn:
            sync_missions(conn, [packets[0]])
            mission_id = packets[0]['mission_id']
            transition_mission(conn, mission_id, 'claimed', 'workflow-test-user')
            transition_mission(conn, mission_id, 'in_progress', 'workflow-test-user')
            chain = verify_mission_chain(conn, mission_id)
            invalid_blocked = False
            try:
                transition_mission(conn, mission_id, 'verified', 'workflow-test-user')
            except ValueError:
                invalid_blocked = True
    return {'status':'PASS' if chain['valid'] and invalid_blocked else 'FAIL',
            'hash_chain_valid':chain['valid'], 'invalid_transition_blocked':invalid_blocked,
            'real_candidate_id':packets[0]['candidate_id']}

mission_qa = dry_run_mission_workflow(mission_packets)
repeat_qa = {'status':'PASS','real_pairs_evaluated':len(repeat_results)} if repeat_results else {
    'status':'NEEDS_REAL_INPUT','requirement':'Two confirmed real scans of the same tree/site.'}
capture_qa = {'status':'PASS','decision':capture_result['decision'],
              'overall_score':capture_result['overall_score']} if capture_result else {
    'status':'NEEDS_REAL_INPUT','requirement':'One real phone or webcam photograph.'}
api_qa = {'status':'PASS','health':health_response.json()}

qa_scorecard = {
    'generated_at':utc_now_iso(),
    'scientific_data_policy':'No synthetic map observations or fabricated scan results.',
    'feature_17_field_missions':mission_qa,
    'feature_18_repeat_scan':repeat_qa,
    'feature_19_guided_capture':capture_qa,
    'api_contracts':api_qa,
}
QA_PATH = PATHS.output_dir / 'features_17_19_qa_scorecard.json'
QA_PATH.write_text(json.dumps(qa_scorecard, indent=2))
print(json.dumps(qa_scorecard, indent=2))
print('Saved:', QA_PATH)

In [ ]:
contracts = {
    'versions':FEATURE_VERSIONS,
    'field_mission':{
        'states':list(MISSION_TRANSITIONS),
        'valid_transitions':{k:sorted(v) for k,v in MISSION_TRANSITIONS.items()},
        'gain_is_calibrated_probability':False,
        'required_frontend_fields':['mission_id','objective_type','location','access_status','mission_score',
                                    'estimated_local_confidence_gain_pp','instructions']},
    'repeat_scan':{
        'same_site_requires_confirmation':True,
        'coverage_mae_basis_pp':COVERAGE_HELDOUT_MAE_PP,
        'required_frontend_fields':['comparability','coverage_change','growth_form_check'],
        'causal_attribution_supported':False},
    'guided_capture':{
        'coverage_threshold':COVERAGE_THRESHOLD,
        'general_quality_available_without_model':True,
        'required_frontend_fields':['decision','overall_score','model_status','metrics','warning_codes','instructions']},
}
CONTRACT_PATH = PATHS.output_dir / 'features_17_19_frontend_contracts.json'
CONTRACT_PATH.write_text(json.dumps(contracts, indent=2))
print('Saved integration contracts:', CONTRACT_PATH)

## How to finish the real test

1. If Feature 17 says `NEEDS_REAL_INPUT`, run the existing real-data planner notebook through its export cell and rerun this notebook from **Real-input discovery**.
2. For Feature 18, open `features_17_19_outputs/repeat_scan_manifest_TO_REVIEW.csv`, assign the same `monitoring_site_id` only to photographs confirmed to show the same tree/site, set `site_id_confirmed=True`, save it as `EcoSignal/repeat_scan_manifest.csv`, and rerun the Feature 18 cells.
3. For Feature 19, run `captured_path = take_guided_photo()`, set `REAL_CAPTURE_IMAGE_PATH = str(captured_path)`, and rerun the assessment cell. Confirm the final coverage model is found before judging the lichen-framing layer.
4. Do not integrate a feature merely because its code runs. The final QA scorecard must show `PASS` using real inputs, and the failure gallery should be manually reviewed.

For the CAC demo, show one field mission, one guided capture with actionable feedback, and one repeat-site result that honestly says either “change supported” or “not enough evidence.” The refusal to overclaim is part of EcoSignal's technical strength.